In [8]:
# Imports & Configuration
import os
import numpy as np
import pandas as pd
import librosa
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Deep Learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.regularizers import l2
# Clustering & Evaluation
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, adjusted_rand_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.manifold import TSNE

print("=" * 80)
print("AUTOENCODER-BASED MUSIC CLUSTERING")
print("=" * 80)
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")


# ============================================================================
# CONFIGURATION
# ============================================================================
class Config:
    # Dataset
    AUDIO_DIR = "/kaggle/input/gtzan-dataset-music-genre-classification/Data/genres_original"
    
    # Audio processing
    SAMPLE_RATE = 22050
    DURATION = 30  # seconds
    N_MELS = 128
    HOP_LENGTH = 512
    N_FRAMES = 1292  # ~30 seconds at sr=22050, hop=512
    
    # Autoencoder architecture
    LATENT_DIM = 128  
    
    # Training
    BATCH_SIZE = 32
    EPOCHS = 50
    LEARNING_RATE = 0.001
    VALIDATION_SPLIT = 0.2
    RANDOM_STATE = 42
    
    # Clustering
    N_CLUSTERS = 5

config = Config()





AUTOENCODER-BASED MUSIC CLUSTERING
TensorFlow version: 2.18.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [9]:
# Step 1: Data Loading
# ============================================================================
# STEP 1: DATA LOADING & PREPROCESSING
# ============================================================================

def extract_mel_spectrogram_raw(audio_path, config):
    """
    Extract RAW Mel spectrogram in dB scale (NO normalization)
    Output shape: (n_mels, n_frames)
    """
    try:
        y, sr = librosa.load(
            audio_path,
            sr=config.SAMPLE_RATE,
            duration=config.DURATION
        )

        # Pad audio
        target_len = int(config.SAMPLE_RATE * config.DURATION)
        if len(y) < target_len:
            y = np.pad(y, (0, target_len - len(y)))

        # Mel spectrogram
        mel_spec = librosa.feature.melspectrogram(
            y=y,
            sr=sr,
            n_mels=config.N_MELS,
            hop_length=config.HOP_LENGTH
        )

        # Convert to dB
        mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)

        # Fix time dimension
        if mel_spec_db.shape[1] > config.N_FRAMES:
            mel_spec_db = mel_spec_db[:, :config.N_FRAMES]
        elif mel_spec_db.shape[1] < config.N_FRAMES:
            pad_width = config.N_FRAMES - mel_spec_db.shape[1]
            mel_spec_db = np.pad(
                mel_spec_db,
                ((0, 0), (0, pad_width)),
                mode='constant'
            )

        return mel_spec_db

    except Exception as e:
        print(f"Error processing {audio_path}: {e}")
        return None

def load_dataset(config):
    """Load dataset and extract RAW mel spectrograms (NO normalization)"""

    print("\n" + "=" * 80)
    print("LOADING GTZAN DATASET")
    print("=" * 80)

    genres = sorted([
        d for d in os.listdir(config.AUDIO_DIR)
        if os.path.isdir(os.path.join(config.AUDIO_DIR, d))
    ])

    all_files, all_genres = [], []

    for genre in genres:
        genre_dir = os.path.join(config.AUDIO_DIR, genre)
        genre_files = list(Path(genre_dir).glob("*.wav"))
        all_files.extend(genre_files)
        all_genres.extend([genre] * len(genre_files))

    spectrograms, valid_files, valid_genres = [], [], []

    print("\nExtracting Mel spectrograms...")
    for file_path, genre in tqdm(zip(all_files, all_genres), total=len(all_files)):
        mel = extract_mel_spectrogram_raw(file_path, config)
        if mel is not None:
            spectrograms.append(mel)
            valid_files.append(file_path)
            valid_genres.append(genre)

    X = np.array(spectrograms)   # RAW mel dB
    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(valid_genres)

    print(f"\n✓ Extracted {len(X)} samples")
    print(f"✓ Shape: {X.shape}")

    return X, y, label_encoder, valid_files, valid_genres





In [10]:

def build_autoencoder(input_shape, latent_dim=128, l2_reg=0.0001):
    """
    Build a simplified Autoencoder based on VGGish architecture.
    
    This autoencoder is designed for the GTZAN dataset:
    - Input shape: (128, 1292) - mel-spectrogram (128 mel bins, 1292 frames)
    - Encoder: Inspired by VGGish but significantly reduced
    - Latent space: 128-D embedding (same as original VGGish)
    - Decoder: Symmetric mirror of encoder
    
    Why this design:
    - GTZAN is a small dataset (1000 samples), so we reduce parameters
    - VGGish has ~4.3M parameters, our encoder has ~200K parameters
    - L2 regularization helps prevent overfitting
    - Symmetrical architecture makes it easier to explain to instructors
    
    Args:
        input_shape (tuple): (n_mels, n_frames) = (128, 1292)
        latent_dim (int): Dimension of bottleneck/embedding layer. Default 128 (same as VGGish)
        l2_reg (float): L2 regularization coefficient for Conv2D layers
    
    Returns:
        tuple: (autoencoder_model, encoder_model)
    
    Model Complexity:
    - Encoder: ~200K parameters (vs VGGish's ~4.3M)
    - Decoder: ~200K parameters  
    - Total: ~400K parameters
    """
    
    # ========================================================================
    # ENCODER
    # ========================================================================
    # Input: (batch, 128, 1292) -> Reshape to 4D: (batch, 128, 1292, 1)
    
    encoder_input = layers.Input(shape=input_shape, name='encoder_input')
    
    # Reshape to 4D for Conv2D (batch, height, width, channels)
    x = layers.Reshape((input_shape[0], input_shape[1], 1), name='reshape_4d')(encoder_input)
    
    # ---- CONV BLOCK 1 ----
    # 3x3 conv, stride 1, SAME padding (similar to VGGish)
    x = layers.Conv2D(
        filters=32,
        kernel_size=(3, 3),
        strides=(1, 1),
        padding='same',
        activation='relu',
        kernel_regularizer=l2(l2_reg),
        name='conv1'
    )(x)
    
    # Max pooling 2x2 with stride 2 (similar to VGGish)
    x = layers.MaxPooling2D(
        pool_size=(2, 2),
        strides=(2, 2),
        padding='same',
        name='pool1'
    )(x)
    
    # ---- CONV BLOCK 2 ----
    x = layers.Conv2D(
        filters=64,
        kernel_size=(3, 3),
        strides=(1, 1),
        padding='same',
        activation='relu',
        kernel_regularizer=l2(l2_reg),
        name='conv2'
    )(x)
    
    x = layers.MaxPooling2D(
        pool_size=(2, 2),
        strides=(2, 2),
        padding='same',
        name='pool2'
    )(x)
    
    # At this point:
    # Input was (128, 1292, 1)
    # After pool1: (64, 646, 32)
    # After pool2: (32, 323, 64)
    
    # ---- BOTTLENECK / EMBEDDING LAYER ----
    # Flatten
    x = layers.Flatten(name='flatten')(x)
    
    # Fully connected layer to latent space
    # Using tanh activation like standard autoencoders
    encoder_output = layers.Dense(
        units=latent_dim,
        activation='tanh',  # Bounded output for stable training
        kernel_regularizer=l2(l2_reg),
        name='embedding'
    )(x)
    
    encoder = Model(encoder_input, encoder_output, name='encoder')
    
    # ========================================================================
    # DECODER (Symmetric Mirror)
    # ========================================================================
    
    decoder_input = layers.Input(shape=(latent_dim,), name='decoder_input')
    
    # First fully connected layer to reconstruct the shape before flattening
    # Need to reconstruct (32, 323, 64) = 663,552 elements
    reconstruct_shape = 32 * 323 * 64
    
    y = layers.Dense(
        units=reconstruct_shape,
        activation='relu',
        kernel_regularizer=l2(l2_reg),
        name='fc_reconstruct'
    )(decoder_input)
    
    # Reshape back to 4D
    y = layers.Reshape((32, 323, 64), name='reshape_for_deconv1')(y)
    
    # ---- DECONV BLOCK 1 (Mirror of pool2 + conv2) ----
    # Upsampling 2x2
    y = layers.UpSampling2D(
        size=(2, 2),
        interpolation='nearest',
        name='upsample1'
    )(y)
    # After upsample1: (64, 646, 64)
    
    # Conv2D to decode
    y = layers.Conv2D(
        filters=32,
        kernel_size=(3, 3),
        strides=(1, 1),
        padding='same',
        activation='relu',
        kernel_regularizer=l2(l2_reg),
        name='deconv1'
    )(y)
    
    # ---- DECONV BLOCK 2 (Mirror of pool1 + conv1) ----
    # Upsampling 2x2
    y = layers.UpSampling2D(
        size=(2, 2),
        interpolation='nearest',
        name='upsample2'
    )(y)
    # After upsample2: (128, 1292, 32)
    
    # Final conv layer to reconstruct original spectrogram
    y = layers.Conv2D(
        filters=1,
        kernel_size=(3, 3),
        strides=(1, 1),
        padding='same',
        activation='linear',  # Linear for regression task (reconstructing mel spectrogram)
        kernel_regularizer=l2(l2_reg),
        name='deconv_final'
    )(y)
    
    # Reshape back to 2D
    decoder_output = layers.Reshape(
        (input_shape[0], input_shape[1]),
        name='decoder_output'
    )(y)
    
    decoder = Model(decoder_input, decoder_output, name='decoder')
    
    # ========================================================================
    # FULL AUTOENCODER
    # ========================================================================
    
    autoencoder_input = layers.Input(shape=input_shape, name='autoencoder_input')
    encoded = encoder(autoencoder_input)
    decoded = decoder(encoded)
    
    autoencoder = Model(autoencoder_input, decoded, name='autoencoder')
    
    return autoencoder, encoder

In [11]:
# Step 3: Train Autoencoder
# ============================================================================
# STEP 3: TRAIN AUTOENCODER
# ============================================================================

def train_autoencoder(X_train, X_val, config):
    """Train the autoencoder"""
    
    print("\n" + "=" * 80)
    print("BUILDING & TRAINING AUTOENCODER")
    print("=" * 80)
    
    # Build model
    input_shape = (config.N_MELS, config.N_FRAMES)
    autoencoder, encoder = build_autoencoder(input_shape, config.LATENT_DIM)
    
    # Print model summary
    print("\n📊 Autoencoder Architecture:")
    autoencoder.summary()
    
    print(f"\n📊 Encoder Architecture:")
    encoder.summary()
    
    # Count parameters
    total_params = autoencoder.count_params()
    encoder_params = encoder.count_params()
    print(f"\n✓ Total parameters: {total_params:,}")
    print(f"✓ Encoder parameters: {encoder_params:,}")
    
    # Compile
    autoencoder.compile(
        optimizer=keras.optimizers.Adam(learning_rate=config.LEARNING_RATE),
        loss='mse',
        metrics=['mae']
    )
    
    # Callbacks
    callbacks = [
        EarlyStopping(
            monitor='val_loss',
            patience=10,
            restore_best_weights=True,
            verbose=1
        ),
        ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=4,
            min_lr=1e-6,
            verbose=1
        ),
        ModelCheckpoint(
            'best_autoencoder.keras',
            monitor='val_loss',
            save_best_only=True,
            verbose=0
        )
    ]
    
    # Train
    print("\n🚀 Starting training...")
    print(f"Training samples: {len(X_train)}")
    print(f"Validation samples: {len(X_val)}")
    
    history = autoencoder.fit(
        X_train, X_train,  # Autoencoder: input = output
        validation_data=(X_val, X_val),
        epochs=config.EPOCHS,
        batch_size=config.BATCH_SIZE,
        callbacks=callbacks,
        verbose=1
    )
    
    print("\n✅ Training completed!")
    
    return autoencoder, encoder, history




In [12]:
# Step 4-9: Analysis & Visualization

# ============================================================================
# STEP 4: VISUALIZE TRAINING
# ============================================================================

def plot_training_history(history):
    """Plot training history"""
    
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Loss
    axes[0].plot(history.history['loss'], label='Train Loss', linewidth=2)
    axes[0].plot(history.history['val_loss'], label='Val Loss', linewidth=2)
    axes[0].set_xlabel('Epoch', fontsize=12)
    axes[0].set_ylabel('Loss (MSE)', fontsize=12)
    axes[0].set_title('Training & Validation Loss', fontsize=14, fontweight='bold')
    axes[0].legend(fontsize=11)
    axes[0].grid(True, alpha=0.3)
    
    # MAE
    axes[1].plot(history.history['mae'], label='Train MAE', linewidth=2)
    axes[1].plot(history.history['val_mae'], label='Val MAE', linewidth=2)
    axes[1].set_xlabel('Epoch', fontsize=12)
    axes[1].set_ylabel('MAE', fontsize=12)
    axes[1].set_title('Training & Validation MAE', fontsize=14, fontweight='bold')
    axes[1].legend(fontsize=11)
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('./training_history.png', dpi=150, bbox_inches='tight')
    plt.close()
    
    print("✓ Training history saved")
# ============================================================================
# STEP 5: EXTRACT EMBEDDINGS
# ============================================================================

def extract_embeddings(encoder, X, batch_size=32):
    """Extract embeddings using trained encoder"""
    
    print("\n" + "=" * 80)
    print("EXTRACTING EMBEDDINGS")
    print("=" * 80)
    
    embeddings = encoder.predict(X, batch_size=batch_size, verbose=1)
    
    print(f"\n✓ Embeddings shape: {embeddings.shape}")
    print(f"✓ Mean: {embeddings.mean():.4f}, Std: {embeddings.std():.4f}")
    print(f"✓ Min: {embeddings.min():.4f}, Max: {embeddings.max():.4f}")
    
    return embeddings


# ============================================================================
# STEP 6: CLUSTERING & EVALUATION
# ============================================================================

def perform_clustering(embeddings, y_true, n_clusters=5):
    """Perform K-Means clustering and evaluate"""
    
    print("\n" + "=" * 80)
    print("K-MEANS CLUSTERING")
    print("=" * 80)
    
    # K-Means
    print(f"\nRunning K-Means with k={n_clusters}...")
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=20)
    clusters = kmeans.fit_predict(embeddings)
    
    # Metrics
    silhouette = silhouette_score(embeddings, clusters)
    davies_bouldin = davies_bouldin_score(embeddings, clusters)
    ari = adjusted_rand_score(y_true, clusters)
    
    print("\n📊 Clustering Results:")
    print(f"  Silhouette Score: {silhouette:.4f} (higher is better, max=1)")
    print(f"  Davies-Bouldin Index: {davies_bouldin:.4f} (lower is better)")
    print(f"  Adjusted Rand Index: {ari:.4f} (higher is better, max=1)")
    
    # Cluster sizes
    unique, counts = np.unique(clusters, return_counts=True)
    print(f"\n  Cluster sizes: {dict(zip(unique, counts))}")
    
    return clusters, {
        'silhouette': silhouette,
        'davies_bouldin': davies_bouldin,
        'ari': ari
    }





In [13]:
"""Main execution pipeline"""

print("\n" + "=" * 80)
print("STARTING AUTOENCODER MUSIC CLUSTERING PIPELINE")
print("=" * 80)

# Set random seeds
np.random.seed(config.RANDOM_STATE)
tf.random.set_seed(config.RANDOM_STATE)

# Step 1: Load dataset
X, y, label_encoder, files, genres = load_dataset(config)

# Step 2: Split data (80-20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=config.RANDOM_STATE, stratify=y
)

print("\n" + "=" * 80)
print("DATA SPLIT (80-20)")
print("=" * 80)
print(f"Training set: {len(X_train)} samples ({len(X_train)/len(X)*100:.1f}%)")
print(f"Test set: {len(X_test)} samples ({len(X_test)/len(X)*100:.1f}%)")

# Step 2.5: Normalize using TRAIN statistics ONLY
print("\n" + "=" * 80)
print("NORMALIZING DATA (TRAIN STATISTICS)")
print("=" * 80)

train_mean = X_train.mean()  
train_std  = X_train.std()

X_train = (X_train - train_mean) / (train_std + 1e-8)
X_test  = (X_test  - train_mean) / (train_std + 1e-8)

# print(f"✓ Train mean: mean={train_mean.mean():.4f}, std={train_mean.std():.4f}")
# print(f"✓ Train std : mean={train_std.mean():.4f}, std={train_std.std():.4f}")
# print(f"✓ Mean shape: {train_mean.shape}")
# print(f"✓ Std shape : {train_std.shape}")

# Step 3: Train autoencoder
autoencoder, encoder, history = train_autoencoder(X_train, X_test, config)

# Step 4: Visualize training
plot_training_history(history)


# Step 10: Also extract embeddings on FULL dataset for complete analysis
print("\n" + "=" * 80)
print("EXTRACTING EMBEDDINGS ON FULL DATASET")
print("=" * 80)

X_full_norm = (X - train_mean) / (train_std + 1e-8)
embeddings_full = extract_embeddings(encoder, X_full_norm)
clusters_full, metrics_full = perform_clustering(embeddings_full, y, config.N_CLUSTERS)

print("\n" + "=" * 80)extract_embeddings
print("FULL DATASET RESULTS")
print("=" * 80)
print(f"  Silhouette Score: {metrics_full['silhouette']:.4f}")
print(f"  Davies-Bouldin Index: {metrics_full['davies_bouldin']:.4f}")
print(f"  Adjusted Rand Index: {metrics_full['ari']:.4f}")




STARTING AUTOENCODER MUSIC CLUSTERING PIPELINE

LOADING GTZAN DATASET

Extracting Mel spectrograms...


 53%|█████▎    | 529/1000 [00:24<00:30, 15.44it/s]

Error processing /kaggle/input/gtzan-dataset-music-genre-classification/Data/genres_original/jazz/jazz.00054.wav: 


100%|██████████| 1000/1000 [00:46<00:00, 21.61it/s]



✓ Extracted 999 samples
✓ Shape: (999, 128, 1292)

DATA SPLIT (80-20)
Training set: 799 samples (80.0%)
Test set: 200 samples (20.0%)

NORMALIZING DATA (TRAIN STATISTICS)

BUILDING & TRAINING AUTOENCODER


...


✓ Total parameters: 170,044,225
✓ Encoder parameters: 84,691,456

🚀 Starting training...
Training samples: 799
Validation samples: 200
Epoch 1/50


I0000 00:00:1768380298.691143     137 service.cc:148] XLA service 0x7bf518004910 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1768380298.691971     137 service.cc:156]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1768380299.076039     137 cuda_dnn.cc:529] Loaded cuDNN version 90300


 1/25 ━━━━━━━━━━━━━━━━━━━━ 7:35 19s/step - loss: 1.2183 - mae: 0.8642

I0000 00:00:1768380314.839403     137 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


25/25 ━━━━━━━━━━━━━━━━━━━━ 53s 1s/step - loss: 0.8181 - mae: 0.6669 - val_loss: 0.5408 - val_mae: 0.5187 - learning_rate: 0.0010
Epoch 2/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 15s 598ms/step - loss: 0.5197 - mae: 0.5114 - val_loss: 0.5280 - val_mae: 0.5173 - learning_rate: 0.0010
Epoch 3/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 14s 579ms/step - loss: 0.5126 - mae: 0.5083 - val_loss: 0.5118 - val_mae: 0.5082 - learning_rate: 0.0010
Epoch 4/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 14s 587ms/step - loss: 0.4991 - mae: 0.5018 - val_loss: 0.5082 - val_mae: 0.5093 - learning_rate: 0.0010
Epoch 5/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 14s 594ms/step - loss: 0.4967 - mae: 0.5050 - val_loss: 0.4983 - val_mae: 0.5092 - learning_rate: 0.0010
Epoch 6/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 4s 174ms/step - loss: 0.4847 - mae: 0.5021 - val_loss: 0.4986 - val_mae: 0.5106 - learning_rate: 0.0010
Epoch 7/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 14s 593ms/step - loss: 0.4829 - mae: 0.5036 - val_loss: 0.4797 - val_mae: 0.5041 - learning_rate: 0.0010
Epoch 8/50
25/

In [14]:
"""
Hàm trực quan hóa t-SNE đơn giản cho kết quả clustering
"""

import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE


def visualize_tsne_clustering(embeddings, clusters, y_true=None, label_names=None, 
                               title='t-SNE Visualization', save_path='tsne_plot.png'):
    """
    Trực quan hóa kết quả clustering bằng t-SNE
    
    Parameters:
    -----------
    embeddings : array (n_samples, n_features)
        Vector embeddings từ model
    clusters : array (n_samples,)
        Nhãn cluster đã predict (0, 1, 2, ...)
    y_true : array, optional
        Nhãn thật (ground truth) nếu có
    label_names : list, optional
        Tên các nhãn thật (ví dụ: ['blues', 'classical', ...])
    title : str
        Tiêu đề của plot
    save_path : str
        Đường dẫn lưu file ảnh
        
    Returns:
    --------
    embeddings_2d : array (n_samples, 2)
        Tọa độ 2D sau khi giảm chiều bằng t-SNE
    """
    
    print("\n" + "="*60)
    print("TRỰC QUAN HÓA T-SNE")
    print("="*60)
    
    # Bước 1: Giảm chiều xuống 2D bằng t-SNE
    print("\n[1/3] Đang chạy t-SNE...")
    tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
    embeddings_2d = tsne.fit_transform(embeddings)
    print("✓ Hoàn thành t-SNE")
    
    # Bước 2: Tạo plot
    print("\n[2/3] Đang vẽ biểu đồ...")
    
    # Nếu có ground truth, vẽ 2 plots
    if y_true is not None:
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        
        # Plot 1: Predicted Clusters
        scatter1 = axes[0].scatter(
            embeddings_2d[:, 0], 
            embeddings_2d[:, 1],
            c=clusters,
            cmap='tab10',
            alpha=0.7,
            s=50,
            edgecolors='black',
            linewidth=0.5
        )
        axes[0].set_title('Predicted Clusters', fontsize=14, fontweight='bold')
        axes[0].set_xlabel('t-SNE 1', fontsize=12)
        axes[0].set_ylabel('t-SNE 2', fontsize=12)
        axes[0].grid(True, alpha=0.3)
        plt.colorbar(scatter1, ax=axes[0], label='Cluster')
        
        # Plot 2: True Labels
        scatter2 = axes[1].scatter(
            embeddings_2d[:, 0], 
            embeddings_2d[:, 1],
            c=y_true,
            cmap='tab10',
            alpha=0.7,
            s=50,
            edgecolors='black',
            linewidth=0.5
        )
        axes[1].set_title('True Labels', fontsize=14, fontweight='bold')
        axes[1].set_xlabel('t-SNE 1', fontsize=12)
        axes[1].set_ylabel('t-SNE 2', fontsize=12)
        axes[1].grid(True, alpha=0.3)
        
        # Thêm legend nếu có tên nhãn
        if label_names is not None:
            handles = []
            for i, name in enumerate(label_names):
                handles.append(plt.Line2D([0], [0], marker='o', color='w',
                                         markerfacecolor=plt.cm.tab10(i),
                                         markersize=8, label=name,
                                         markeredgecolor='black', markeredgewidth=0.5))
            axes[1].legend(handles=handles, loc='best', fontsize=9)
        else:
            plt.colorbar(scatter2, ax=axes[1], label='Label')
        
        fig.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
        
    else:
        # Chỉ vẽ 1 plot cho clusters
        fig, ax = plt.subplots(figsize=(10, 8))
        
        scatter = ax.scatter(
            embeddings_2d[:, 0], 
            embeddings_2d[:, 1],
            c=clusters,
            cmap='tab10',
            alpha=0.7,
            s=50,
            edgecolors='black',
            linewidth=0.5
        )
        ax.set_title(title, fontsize=16, fontweight='bold')
        ax.set_xlabel('t-SNE Component 1', fontsize=12)
        ax.set_ylabel('t-SNE Component 2', fontsize=12)
        ax.grid(True, alpha=0.3)
        plt.colorbar(scatter, label='Cluster')
    
    # Bước 3: Lưu file
    print("\n[3/3] Đang lưu file...")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"✓ Đã lưu: {save_path}")
    print("="*60)
    
    return embeddings_2d


In [15]:
visualize_tsne_clustering(embeddings=embeddings_full,         
    clusters=clusters_full,    
    title='K-Means Clustering Results',
    save_path='tsne_kmeans.png')


TRỰC QUAN HÓA T-SNE

[1/3] Đang chạy t-SNE...
✓ Hoàn thành t-SNE

[2/3] Đang vẽ biểu đồ...

[3/3] Đang lưu file...
✓ Đã lưu: tsne_kmeans.png


array([[ 53.241673 ,  10.7554455],
       [ 36.64581  ,  -2.239765 ],
       [ 40.241676 ,  -2.3587687],
       ...,
       [ 25.700191 ,   8.642904 ],
       [-46.000896 ,  -7.0605516],
       [-30.622131 ,  17.898285 ]], dtype=float32)